# download the data

In [2]:
import os
import pandas as pd
import opendatasets as od

# Configuration
dataset_url = "https://www.kaggle.com/datasets/osmi/mental-health-in-tech-survey"
data_dir = "./mental-health-in-tech-survey"
file_path = f"{data_dir}/survey.csv"

# 1. Check if the directory and file already exist
if not os.path.exists(file_path):
    print("Data not found. Downloading from Kaggle...")
    # This will prompt for your kaggle.json credentials if not found locally
    od.download(dataset_url)
else:
    print("Data already exists. Skipping download.")

# 2. Load the data into a pandas dataframe
survey_data = pd.read_csv(file_path)

# 3. Quick sanity check on the 'comments' column
# Many entries are NaN, so we'll filter those for your analysis
text_responses = survey_data.dropna(subset=['comments'])

print(f"Total rows in dataset: {len(survey_data)}")
print(f"Rows with text comments for analysis: {len(text_responses)}")

Data already exists. Skipping download.
Total rows in dataset: 1259
Rows with text comments for analysis: 164


# test of hdbscan

In [5]:
from sentence_transformers import SentenceTransformer
import umap
import hdbscan

# 1. Prepare data safely
analysis_df = survey_data.dropna(subset=['comments']).copy()
comments_list = analysis_df['comments'].tolist()

# 2. Generate Embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(comments_list, show_progress_bar=True)

# 3. Reduce Dimensions (The "Knowledge Discovery" Secret Sauce)
# Reducing to 5 dimensions helps HDBSCAN find density without losing the 'meaning'
umap_reducer = umap.UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=42)
reduced_embeddings = umap_reducer.fit_transform(embeddings)

# 4. Cluster with HDBSCAN
clusterer = hdbscan.HDBSCAN(min_cluster_size=5, min_samples=2, gen_min_span_tree=True)
analysis_df['cluster_id'] = clusterer.fit_predict(reduced_embeddings)

print(f"Discovered {analysis_df['cluster_id'].max() + 1} distinct meta-categories!")

Loading weights: 100%|█████████████████████| 103/103 [00:00<00:00, 445.35it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|███████████████████████████████████████████████████████████████████████████| 6/6 [00:02<00:00,  2.18it/s]
C:\Anaconda3\envs\text_analysis\lib\site-packages\umap\umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


Discovered 14 distinct meta-categories!
